# Advanced Semantic Search Demo with MiniLM and RAG

This notebook demonstrates semantic search using both MiniLM embeddings and Retrieval-Augmented Generation (RAG) for enhanced search capabilities.

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel, pipeline
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
import faiss

# Load preprocessed data
train_df = pd.read_csv('data/train_preprocessed.csv')

# Initialize MiniLM model and tokenizer
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Function to get embeddings
def get_embeddings(texts, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors='pt', max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state.mean(dim=1)
            embeddings.append(batch_embeddings)
    return torch.cat(embeddings, dim=0)

# Get embeddings for all documents
document_embeddings = get_embeddings(train_df['processed_text'].tolist())

# Create FAISS index for MiniLM
dimension = document_embeddings.shape[1]
minilm_index = faiss.IndexFlatL2(dimension)
minilm_index.add(document_embeddings.numpy())

# Function for MiniLM semantic search
def minilm_semantic_search(query, k=5):
    query_embedding = get_embeddings([query])
    distances, indices = minilm_index.search(query_embedding.numpy(), k)
    
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'document': train_df['processed_text'].iloc[idx],
            'label': train_df['label'].iloc[idx],
            'distance': distances[0][i]
        })
    return results

# Set up RAG components
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Split documents into chunks
texts = text_splitter.split_text('\n'.join(train_df['processed_text'].tolist()))

# Initialize embeddings for RAG
embeddings = HuggingFaceEmbeddings(model_name=model_name)

# Create vector store
vectorstore = FAISS.from_texts(texts, embeddings)

# Initialize LLM for RAG
llm = HuggingFacePipeline.from_model_id(
    model_id="gpt2",
    task="text-generation",
    pipeline_kwargs={"max_length": 100}
)

# Create RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

# Function for RAG-based search
def rag_semantic_search(query, k=5):
    # Get relevant documents using RAG
    relevant_docs = vectorstore.similarity_search(query, k=k)
    
    # Generate answers using RAG chain
    answer = rag_chain.run(query)
    
    return {
        'answer': answer,
        'relevant_documents': relevant_docs
    }

# Demo function to compare both approaches
def compare_search_methods(query):
    print(f"\nQuery: {query}")
    
    print("\nMiniLM Semantic Search Results:")
    minilm_results = minilm_semantic_search(query)
    for i, result in enumerate(minilm_results, 1):
        print(f"\n{i}. Label: {result['label']}")
        print(f"Distance: {result['distance']:.4f}")
        print(f"Text preview: {result['document'][:200]}...")
    
    print("\nRAG-based Search Results:")
    rag_results = rag_semantic_search(query)
    print(f"\nGenerated Answer: {rag_results['answer']}")
    print("\nRelevant Documents:")
    for i, doc in enumerate(rag_results['relevant_documents'], 1):
        print(f"\n{i}. {doc.page_content[:200]}...")

# Run demo comparisons
demo_queries = [
    "What are the latest developments in technology?",
    "How is the current state of the financial markets?",
    "What are the major political events happening?"
]

for query in demo_queries:
    compare_search_methods(query)

# Save search results
search_results = {
    'queries': demo_queries,
    'minilm_results': {
        query: minilm_semantic_search(query) for query in demo_queries
    },
    'rag_results': {
        query: rag_semantic_search(query) for query in demo_queries
    }
}

import json
with open('data/advanced_search_results.json', 'w') as f:
    json.dump(search_results, f)